In [ ]:
# Install the Python packages required by this notebook.
%pip install -q duckdb pyarrow scikit-learn xgboost scipy joblib

In [ ]:
# Mount Google Drive so this notebook can access the private MIMIC-IV data and derived files.
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# Define the MIMIC-IV folders and the shared derived-data folder used by all notebooks.
from pathlib import Path

DATA_ROOT = Path("/content/drive/MyDrive/Early Acute Kidney Injury Prediction + Production Monitoring/data")
HOSP_DIR = DATA_ROOT / "hosp"
ICU_DIR = DATA_ROOT / "icu"
DERIVED_DIR = DATA_ROOT / "derived"
DERIVED_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
# Import the libraries used to build the creatinine and urine-output AKI labels.
import duckdb
import numpy as np
import pandas as pd

In [ ]:
# Load the eligible ICU cohort created by Notebook 01 and restore its datetime columns.
eligible_cohort = pd.read_parquet(DERIVED_DIR / "eligible_cohort.parquet")

for col in ["intime", "outtime", "prediction_time", "prediction_end", "admittime", "dischtime"]:
    if col in eligible_cohort.columns:
        eligible_cohort[col] = pd.to_datetime(eligible_cohort[col])

print("Eligible ICU stays:", f"{len(eligible_cohort):,}")
eligible_cohort.head()

In [ ]:
# Define the MIMIC-IV item IDs used for serum creatinine, urine output, and patient weight.
CREATININE_ITEMID = 50912

URINE_OUTPUT_ITEMIDS = [
    226559, 226560, 226561, 226584, 226563,
    226564, 226565, 226567, 226557, 226558,
    227488, 227489,
]

WEIGHT_ITEMIDS = [226512, 224639]

In [ ]:
# Extract serum creatinine measurements from seven days before ICU admission through the prediction horizon.
labevents_path = str(HOSP_DIR / "labevents.csv.gz")

con = duckdb.connect()
con.register(
    "cohort_df",
    eligible_cohort[
        ["subject_id", "hadm_id", "stay_id", "intime", "outtime", "prediction_time", "prediction_end"]
    ]
)

creatinine = con.execute(f"""
    SELECT
        c.subject_id,
        c.hadm_id,
        c.stay_id,
        c.intime,
        c.outtime,
        c.prediction_time,
        c.prediction_end,
        CAST(le.charttime AS TIMESTAMP) AS charttime,
        AVG(CAST(le.valuenum AS DOUBLE)) AS creatinine
    FROM cohort_df c
    INNER JOIN read_csv_auto('{labevents_path}') le
        ON c.subject_id = le.subject_id
       AND le.itemid = {CREATININE_ITEMID}
       AND le.valuenum IS NOT NULL
       AND CAST(le.valuenum AS DOUBLE) > 0
       AND CAST(le.valuenum AS DOUBLE) <= 150
       AND CAST(le.charttime AS TIMESTAMP) >= c.intime - INTERVAL '7 days'
       AND CAST(le.charttime AS TIMESTAMP) <= LEAST(c.outtime, c.prediction_end)
    GROUP BY
        c.subject_id,
        c.hadm_id,
        c.stay_id,
        c.intime,
        c.outtime,
        c.prediction_time,
        c.prediction_end,
        CAST(le.charttime AS TIMESTAMP)
    ORDER BY c.stay_id, charttime
""").df()

con.close()

print("Creatinine measurements:", f"{len(creatinine):,}")
creatinine.head()

In [ ]:
# Calculate each creatinine value's lowest prior measurement within the previous 48 hours and seven days.
creatinine = creatinine.sort_values(["stay_id", "charttime"]).reset_index(drop=True)

def add_creatinine_baselines(group):
    group = group.sort_values("charttime").copy()
    times = group["charttime"]
    values = group["creatinine"].to_numpy()
    low_48h = []
    low_7d = []

    for i, current_time in enumerate(times):
        previous_times = times.iloc[:i]
        previous_values = values[:i]

        mask_48h = previous_times >= current_time - pd.Timedelta(hours=48)
        mask_7d = previous_times >= current_time - pd.Timedelta(days=7)

        low_48h.append(previous_values[mask_48h.to_numpy()].min() if mask_48h.any() else np.nan)
        low_7d.append(previous_values[mask_7d.to_numpy()].min() if mask_7d.any() else np.nan)

    group["creatinine_low_48h"] = low_48h
    group["creatinine_low_7d"] = low_7d
    return group

creatinine = (
    creatinine
    .groupby("stay_id", group_keys=False)
    .apply(add_creatinine_baselines)
    .reset_index(drop=True)
)

creatinine.head()

In [ ]:
# Assign the creatinine-based KDIGO AKI stage at every creatinine measurement.
def creatinine_stage(row):
    current = row["creatinine"]
    low_48h = row["creatinine_low_48h"]
    low_7d = row["creatinine_low_7d"]

    if pd.notna(low_7d) and current >= 3.0 * low_7d:
        return 3

    if current >= 4.0 and (
        (pd.notna(low_48h) and current >= low_48h + 0.3)
        or (pd.notna(low_7d) and current >= 1.5 * low_7d)
    ):
        return 3

    if pd.notna(low_7d) and current >= 2.0 * low_7d:
        return 2

    if pd.notna(low_48h) and current >= low_48h + 0.3:
        return 1

    if pd.notna(low_7d) and current >= 1.5 * low_7d:
        return 1

    return 0

creatinine["aki_stage_creatinine"] = creatinine.apply(creatinine_stage, axis=1)
creatinine["aki_stage_creatinine"].value_counts().sort_index()

In [ ]:
# Extract the first valid ICU weight available by the 12-hour prediction time for each stay.
chartevents_path = str(ICU_DIR / "chartevents.csv.gz")

con = duckdb.connect()
con.register(
    "cohort_df",
    eligible_cohort[["stay_id", "intime", "prediction_time"]]
)

weight_ids_sql = ", ".join(str(x) for x in WEIGHT_ITEMIDS)

weights = con.execute(f"""
    WITH ranked_weights AS (
        SELECT
            c.stay_id,
            CAST(ce.charttime AS TIMESTAMP) AS charttime,
            ce.itemid,
            CAST(ce.valuenum AS DOUBLE) AS weight_kg,
            ROW_NUMBER() OVER (
                PARTITION BY c.stay_id
                ORDER BY
                    CASE WHEN ce.itemid = 226512 THEN 0 ELSE 1 END,
                    CAST(ce.charttime AS TIMESTAMP)
            ) AS rn
        FROM cohort_df c
        INNER JOIN read_csv_auto('{chartevents_path}') ce
            ON c.stay_id = ce.stay_id
           AND ce.itemid IN ({weight_ids_sql})
           AND ce.valuenum IS NOT NULL
           AND CAST(ce.valuenum AS DOUBLE) BETWEEN 20 AND 400
           AND CAST(ce.charttime AS TIMESTAMP) <= c.prediction_time
    )
    SELECT stay_id, weight_kg
    FROM ranked_weights
    WHERE rn = 1
""").df()

con.close()

print("Stays with weight:", f"{len(weights):,}")
weights.head()

In [ ]:
# Extract urine-output events from ICU admission through the end of the 24-hour prediction horizon.
outputevents_path = str(ICU_DIR / "outputevents.csv.gz")

con = duckdb.connect()
con.register(
    "cohort_df",
    eligible_cohort[["stay_id", "intime", "outtime", "prediction_time", "prediction_end"]]
)

urine_ids_sql = ", ".join(str(x) for x in URINE_OUTPUT_ITEMIDS)

urine = con.execute(f"""
    SELECT
        c.stay_id,
        c.intime,
        c.outtime,
        c.prediction_time,
        c.prediction_end,
        CAST(oe.charttime AS TIMESTAMP) AS charttime,
        SUM(
            CASE
                WHEN oe.itemid = 227488 AND CAST(oe.value AS DOUBLE) > 0
                    THEN -1.0 * CAST(oe.value AS DOUBLE)
                ELSE CAST(oe.value AS DOUBLE)
            END
        ) AS urine_ml
    FROM cohort_df c
    INNER JOIN read_csv_auto('{outputevents_path}') oe
        ON c.stay_id = oe.stay_id
       AND oe.itemid IN ({urine_ids_sql})
       AND oe.value IS NOT NULL
       AND CAST(oe.charttime AS TIMESTAMP) >= c.intime
       AND CAST(oe.charttime AS TIMESTAMP) <= LEAST(c.outtime, c.prediction_end)
    GROUP BY
        c.stay_id,
        c.intime,
        c.outtime,
        c.prediction_time,
        c.prediction_end,
        CAST(oe.charttime AS TIMESTAMP)
    ORDER BY c.stay_id, charttime
""").df()

con.close()

urine = urine.merge(weights, on="stay_id", how="left")
print("Urine-output events:", f"{len(urine):,}")
urine.head()

In [ ]:
# Calculate rolling 6-hour, 12-hour, and 24-hour urine-output rates in milliliters per kilogram per hour.
def add_urine_windows(group):
    group = group.sort_values("charttime").copy()
    series = group.set_index("charttime")["urine_ml"]

    group["urine_6h_ml"] = series.rolling("6h", closed="both").sum().to_numpy()
    group["urine_12h_ml"] = series.rolling("12h", closed="both").sum().to_numpy()
    group["urine_24h_ml"] = series.rolling("24h", closed="both").sum().to_numpy()

    elapsed_hours = (
        (group["charttime"] - group["intime"]).dt.total_seconds() / 3600.0
    ).clip(lower=0)

    group["observed_hours_6h"] = np.minimum(elapsed_hours, 6.0)
    group["observed_hours_12h"] = np.minimum(elapsed_hours, 12.0)
    group["observed_hours_24h"] = np.minimum(elapsed_hours, 24.0)

    group["urine_rate_6h"] = group["urine_6h_ml"] / (
        group["weight_kg"] * group["observed_hours_6h"].replace(0, np.nan)
    )
    group["urine_rate_12h"] = group["urine_12h_ml"] / (
        group["weight_kg"] * group["observed_hours_12h"].replace(0, np.nan)
    )
    group["urine_rate_24h"] = group["urine_24h_ml"] / (
        group["weight_kg"] * group["observed_hours_24h"].replace(0, np.nan)
    )

    return group

urine = (
    urine
    .groupby("stay_id", group_keys=False)
    .apply(add_urine_windows)
    .reset_index(drop=True)
)

urine.head()

In [ ]:
# Assign the urine-output-based KDIGO AKI stage at every available urine-output time point.
def urine_stage(row):
    if pd.isna(row["weight_kg"]):
        return np.nan

    if row["observed_hours_24h"] >= 24 and row["urine_rate_24h"] < 0.3:
        return 3

    if row["observed_hours_12h"] >= 12 and row["urine_12h_ml"] <= 0:
        return 3

    if row["observed_hours_12h"] >= 12 and row["urine_rate_12h"] < 0.5:
        return 2

    if row["observed_hours_6h"] >= 6 and row["urine_rate_6h"] < 0.5:
        return 1

    return 0

urine["aki_stage_urine"] = urine.apply(urine_stage, axis=1)
urine["aki_stage_urine"].value_counts(dropna=False).sort_index()

In [ ]:
# Combine creatinine and urine-output stages into one event timeline for each ICU stay.
creatinine_events = creatinine[
    ["stay_id", "charttime", "prediction_time", "prediction_end", "aki_stage_creatinine"]
].rename(columns={"aki_stage_creatinine": "aki_stage"})
creatinine_events["source"] = "creatinine"

urine_events = urine[
    ["stay_id", "charttime", "prediction_time", "prediction_end", "aki_stage_urine"]
].rename(columns={"aki_stage_urine": "aki_stage"})
urine_events["source"] = "urine_output"

aki_events = pd.concat(
    [creatinine_events, urine_events],
    ignore_index=True
).dropna(subset=["aki_stage"])

aki_events["aki_stage"] = aki_events["aki_stage"].astype(int)
aki_events.head()

In [ ]:
# Summarize the maximum AKI stage before prediction and during the following 24 hours for each ICU stay.
before_stage = (
    aki_events[
        aki_events["charttime"] <= aki_events["prediction_time"]
    ]
    .groupby("stay_id")["aki_stage"]
    .max()
    .rename("max_stage_before_prediction")
)

future_stage = (
    aki_events[
        (aki_events["charttime"] > aki_events["prediction_time"])
        & (aki_events["charttime"] <= aki_events["prediction_end"])
    ]
    .groupby("stay_id")["aki_stage"]
    .max()
    .rename("max_stage_next_24h")
)

future_measurement = (
    aki_events[
        (aki_events["charttime"] > aki_events["prediction_time"])
        & (aki_events["charttime"] <= aki_events["prediction_end"])
    ]
    .groupby("stay_id")
    .size()
    .gt(0)
    .rename("future_measurement_available")
)

In [ ]:
# Create the final new Stage 2+ AKI target while excluding prevalent AKI and unobservable negative cases.
labels = eligible_cohort[
    [
        "subject_id",
        "hadm_id",
        "stay_id",
        "outtime",
        "prediction_time",
        "prediction_end",
        "anchor_year_group",
    ]
].copy()

labels = labels.merge(before_stage, on="stay_id", how="left")
labels = labels.merge(future_stage, on="stay_id", how="left")
labels = labels.merge(future_measurement, on="stay_id", how="left")

labels["max_stage_before_prediction"] = labels["max_stage_before_prediction"].fillna(0).astype(int)
labels["max_stage_next_24h"] = labels["max_stage_next_24h"].fillna(0).astype(int)
labels["future_measurement_available"] = labels["future_measurement_available"].fillna(False)

labels["already_stage2_plus"] = labels["max_stage_before_prediction"] >= 2
labels["future_stage2_plus"] = labels["max_stage_next_24h"] >= 2
labels["full_24h_followup"] = labels["outtime"] >= labels["prediction_end"]

labels["label_observable"] = (
    labels["future_stage2_plus"]
    | (labels["full_24h_followup"] & labels["future_measurement_available"])
)

labels["target"] = labels["future_stage2_plus"].astype(int)

labels = labels[
    (~labels["already_stage2_plus"])
    & labels["label_observable"]
].reset_index(drop=True)

print("Final labeled stays:", f"{len(labels):,}")
print("Positive AKI Stage 2+ cases:", f"{labels['target'].sum():,}")
print("Target prevalence:", f"{labels['target'].mean():.3%}")

In [ ]:
# Save the AKI labels and clinical staging timelines for the later notebooks.
labels.to_parquet(DERIVED_DIR / "aki_labels.parquet", index=False)
creatinine.to_parquet(DERIVED_DIR / "creatinine_timeline.parquet", index=False)
urine.to_parquet(DERIVED_DIR / "urine_timeline.parquet", index=False)
aki_events.to_parquet(DERIVED_DIR / "aki_events.parquet", index=False)

print("Saved:", DERIVED_DIR / "aki_labels.parquet")